In [1]:
!pip install datasets
!pip install pyarrow

In [2]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import warnings
import os
from email import message_from_string
warnings.filterwarnings('ignore')

In [3]:
path = r"D:\Northeastern\Fall2025\DS5500\Spam_Email_Detection\raw_data"

In [4]:
def load_raw_datasets(folder_path):
    data_dict = {}
    for file_name in os.listdir(folder_path):
        if file_name.endswith(".csv"):
            file_path = os.path.join(folder_path, file_name)
            try:
                df = pd.read_csv(file_path, engine='python', on_bad_lines='skip')
                data_dict[os.path.splitext(file_name)[0]] = df
                print(f"Loaded: {file_name} → {df.shape[0]} rows, {df.shape[1]} columns")
            except Exception as e:
                print(f"Could not load {file_name}: {e}")
    return data_dict

In [5]:
df_nazario = pd.read_csv(os.path.join(path, "Nazario.csv"))
df_nigerian = pd.read_csv(os.path.join(path, "Nigerian_Fraud.csv"))
df_trec_05 = pd.read_csv(os.path.join(path, "TREC_05.csv"), encoding='latin1', engine='python', on_bad_lines='skip')
df_trec_06 = pd.read_csv(os.path.join(path, "TREC_06.csv"), encoding='latin1', engine='python', on_bad_lines='skip')
df_trec_07 = pd.read_csv(os.path.join(path, "TREC_07.csv"), encoding='latin1', engine='python', on_bad_lines='skip')

In [6]:
df_nazario.head(3)

,sender,receiver,date,subject,body,urls,label
0,Mail System Internal Data <MAILER-DAEMON@monke...,NaN,28 Sep 2017 09:57:25 -0400,DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA,This text is part of the internal format of yo...,1,1
1,cPanel <service@cpanel.com>,jose@monkey.org,"Fri, 30 Oct 2015 00:00:48 -0500",Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1,1
2,Microsoft Outlook <recepcao@unimedceara.com.br>,NaN,"Fri, 30 Oct 2015 06:21:59 -0300 (BRT)",Helpdesk Mailbox Alert!!!,Your two incoming mails were placed on pending...,1,1


In [7]:
df_nigerian.head(3)

,sender,receiver,date,subject,body,urls,label
0,MR. JAMES NGOLA. <james_ngola2002@maktoob.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 02:38:20 +0000",URGENT BUSINESS ASSISTANCE AND PARTNERSHIP,FROM:MR. JAMES NGOLA.\nCONFIDENTIAL TEL: 233-2...,0,1
1,Mr. Ben Suleman <bensul2004nng@spinfinder.com>,R@M,"Thu, 31 Oct 2002 05:10:00 -0000",URGENT ASSISTANCE /RELATIONSHIP (P),"Dear Friend,\n\nI am Mr. Ben Suleman a custom ...",0,1
2,PRINCE OBONG ELEME <obong_715@epatra.com>,webmaster@aclweb.org,"Thu, 31 Oct 2002 22:17:55 +0100",GOOD DAY TO YOU,FROM HIS ROYAL MAJESTY (HRM) CROWN RULER OF EL...,0,1


In [8]:
df_trec_05 = df_trec_05[df_trec_05['label'].astype(str).str.strip().isin(['0', '1'])]
df_trec_05['label'] = df_trec_05['label'].astype(int)
df_trec_05['label'] = (
    df_trec_05['label']
    .fillna(0)               
    .astype(str)             
    .str.strip()             
    .replace('', '0')        
    .astype(int)             
)
df_trec_05.head(3)

,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr PW: bnaweb22 -----O...,0,1
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,0
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,1


In [9]:
df_trec_06.head(3)

,sender,receiver,date,subject,body,label,urls
0,jhpb@sarto.budd-lake.nj.us,NaN,"Tue, 28 Jul 1992 03:13:55 +0000",new Catholic mailing list now up and running,The mailing list I queried about a few weeks a...,0.0,0.0
1,Stella Lowry <rookcuduq@yahoo.com>,Brian <bernice@groucho.cs.psu.edu>,"Sat, 03 Apr 1993 10:34:36 -0500",re[12]:,\n ...,1.0,1.0
2,Walter <trwmpca@downtowncumberland.com>,arline@groucho.cs.psu.edu,"Tue, 06 Apr 1993 20:33:13 -0600",Take a moment to explore this.,Academic Qualifications available from prestig...,1.0,0.0


In [10]:
df_trec_07 = df_trec_07[df_trec_07['label'].astype(str).str.strip().isin(['0', '1'])]
df_trec_07['label'] = df_trec_07['label'].astype(int)
df_trec_07['label'] = (
    df_trec_07['label']
    .fillna(0)               
    .astype(str)             
    .str.strip()             
    .replace('', '0')        
    .astype(int)             
)
df_trec_05.head(3)

,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr PW: bnaweb22 -----O...,0,1
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,0
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,1


In [11]:
def count_email_labels(df, name="Dataset"):
    counts = df['label'].value_counts().to_dict()
    ham_count = counts.get(0, 0)
    spam_count = counts.get(1, 0)
    print(f"{name}\n")
    print(f"  Ham (label=0): {ham_count}")
    print(f"  Spam (label=1): {spam_count}")
    print(f"  Total emails: {ham_count + spam_count}\n")

    return {"ham": ham_count, "spam": spam_count}

In [12]:
count_email_labels(df_nazario, "Nazario")
count_email_labels(df_nigerian, "Nigerian Fraud")
count_email_labels(df_trec_05, "TREC_05")
count_email_labels(df_trec_06, "TREC_06")
count_email_labels(df_trec_07, "TREC_07")

Nazario

  Ham (label=0): 0
  Spam (label=1): 1565
  Total emails: 1565

Nigerian Fraud

  Ham (label=0): 0
  Spam (label=1): 3332
  Total emails: 3332

TREC_05

  Ham (label=0): 32278
  Spam (label=1): 22932
  Total emails: 55210

TREC_06

  Ham (label=0): 12393
  Spam (label=1): 3989
  Total emails: 16382

TREC_07

  Ham (label=0): 24353
  Spam (label=1): 29392
  Total emails: 53745



{'ham': 24353, 'spam': 29392}

In [13]:
df_enron = pd.read_csv(os.path.join(path, "emails.csv"))

In [14]:
df_enron.head(3)

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...


In [15]:
def parse_enron_email(raw_text):
    try:
        msg = message_from_string(raw_text)
        sender = msg.get('From')
        receiver = msg.get('To')
        date = msg.get('Date')
        subject = msg.get('Subject')

        if msg.is_multipart():
            body = ''
            for part in msg.walk():
                content_type = part.get_content_type()
                if content_type == 'text/plain':
                    body += part.get_payload(decode=True).decode(errors='ignore')
        else:
            body = msg.get_payload(decode=True)
            if body:
                body = body.decode(errors='ignore')
            else:
                body = ''

        body = re.sub(r'\s+', ' ', body).strip()

        return pd.Series([sender, receiver, date, subject, body])
    except Exception as e:
        return pd.Series([None, None, None, None, None])

In [16]:
df_parsed = df_enron['message'].apply(parse_enron_email)
df_parsed.columns = ['sender', 'receiver', 'date', 'subject', 'body']

In [17]:
df_parsed['label'] = 0   # all Enron emails are ham
df_parsed['urls'] = df_parsed['body'].str.count(r'http[s]?://')

In [18]:
df_enron_clean = df_parsed[['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']]
df_enron_clean.head(3)

,sender,receiver,date,subject,body,label,urls
0,phillip.allen@enron.com,tim.belden@enron.com,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",,Here is our forecast,0,0
1,phillip.allen@enron.com,john.lavorato@enron.com,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",Re:,Traveling to have a business meeting takes the...,0,0
2,phillip.allen@enron.com,leah.arsdall@enron.com,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",Re: test,test successful. way to go!!!,0,0


In [19]:
df_enron_clean.shape

(517401, 7)

In [20]:
df_nazario = df_nazario[df_nazario['label'] == 1].copy()
df_nigerian = df_nigerian[df_nigerian['label'] == 1].copy()
df_trec_05 = df_trec_05[df_trec_05['label'] == 1].copy()
df_trec_06 = df_trec_06[df_trec_06['label'] == 1].copy()
df_trec_07 = df_trec_07[df_trec_07['label'] == 1].copy()

In [21]:
df_nazario['category'] = 'phishing'
df_nigerian['category'] = 'phishing'
df_trec_05['category'] = 'spam'
df_trec_06['category'] = 'spam'
df_trec_07['category'] = 'spam'
df_enron_clean['category'] = 'legitimate'

In [22]:
combined_df = pd.concat([
    df_nazario,
    df_nigerian,
    df_trec_05,
    df_trec_06,
    df_trec_07,
    df_enron_clean
], ignore_index=True)

In [23]:
combined_df

,sender,receiver,date,subject,body,urls,label,category
0,Mail System Internal Data <MAILER-DAEMON@monke...,NaN,28 Sep 2017 09:57:25 -0400,DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA,This text is part of the internal format of yo...,1,1.0,phishing
1,cPanel <service@cpanel.com>,jose@monkey.org,"Fri, 30 Oct 2015 00:00:48 -0500",Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1,1.0,phishing
2,Microsoft Outlook <recepcao@unimedceara.com.br>,NaN,"Fri, 30 Oct 2015 06:21:59 -0300 (BRT)",Helpdesk Mailbox Alert!!!,Your two incoming mails were placed on pending...,1,1.0,phishing
3,Ann Garcia <AnGarcia@mcoe.org>,"""info@maaaaa.org"" <info@maaaaa.org>","Fri, 30 Oct 2015 14:54:33 +0000",IT-Service Help Desk,Password will expire in 3 days. Click Here To ...,0,1.0,phishing
4,"""USAA"" <usaaacctupdate@sccu4u.com>",Recipients <usaaacctupdate@sccu4u.com>,"Fri, 30 Oct 2015 14:02:33 -0500",Final USAA Reminder - Update Your Account Now,"To ensure delivery to your inbox, please add U...",1,1.0,phishing
...,...,...,...,...,...,...,...,...
578606,john.zufferli@enron.com,kori.loibl@enron.com,"Wed, 28 Nov 2001 13:30:11 -0800 (PST)",Trade with John Lavorato,This is a trade with OIL-SPEC-HEDGE-NG (John L...,0,0.0,legitimate
578607,john.zufferli@enron.com,john.lavorato@enron.com,"Wed, 28 Nov 2001 12:47:48 -0800 (PST)",Gas Hedges,Some of my position is with the Alberta Term b...,0,0.0,legitimate
578608,john.zufferli@enron.com,dawn.doucet@enron.com,"Wed, 28 Nov 2001 07:20:00 -0800 (PST)",RE: CONFIDENTIAL,"2 -----Original Message----- From: Doucet, Daw...",0,0.0,legitimate
578609,john.zufferli@enron.com,jeanie.slone@enron.com,"Tue, 27 Nov 2001 11:52:45 -0800 (PST)",Calgary Analyst/Associate,Analyst Rank Stephane Brodeur 1 Chad Clark 1 I...,0,0.0,legitimate


In [24]:
cols_to_drop = ['label','date']
combined_df.drop(columns=[c for c in cols_to_drop if c in combined_df.columns], inplace=True, errors='ignore')

In [25]:
combined_df

,sender,receiver,subject,body,urls,category
0,Mail System Internal Data <MAILER-DAEMON@monke...,NaN,DON'T DELETE THIS MESSAGE -- FOLDER INTERNAL DATA,This text is part of the internal format of yo...,1,phishing
1,cPanel <service@cpanel.com>,jose@monkey.org,Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1,phishing
2,Microsoft Outlook <recepcao@unimedceara.com.br>,NaN,Helpdesk Mailbox Alert!!!,Your two incoming mails were placed on pending...,1,phishing
3,Ann Garcia <AnGarcia@mcoe.org>,"""info@maaaaa.org"" <info@maaaaa.org>",IT-Service Help Desk,Password will expire in 3 days. Click Here To ...,0,phishing
4,"""USAA"" <usaaacctupdate@sccu4u.com>",Recipients <usaaacctupdate@sccu4u.com>,Final USAA Reminder - Update Your Account Now,"To ensure delivery to your inbox, please add U...",1,phishing
...,...,...,...,...,...,...
578606,john.zufferli@enron.com,kori.loibl@enron.com,Trade with John Lavorato,This is a trade with OIL-SPEC-HEDGE-NG (John L...,0,legitimate
578607,john.zufferli@enron.com,john.lavorato@enron.com,Gas Hedges,Some of my position is with the Alberta Term b...,0,legitimate
578608,john.zufferli@enron.com,dawn.doucet@enron.com,RE: CONFIDENTIAL,"2 -----Original Message----- From: Doucet, Daw...",0,legitimate
578609,john.zufferli@enron.com,jeanie.slone@enron.com,Calgary Analyst/Associate,Analyst Rank Stephane Brodeur 1 Chad Clark 1 I...,0,legitimate


In [26]:
combined_df.to_csv('combined_df.csv', index=False)